Script to calculate max. snow depth for all stations during the 2017/18 winter season
-> for Table A1 in paper

In [ ]:
import numpy as np
from pathlib import Path
import platform
import pandas as pd
import glob
import os

if platform.system() == 'Windows':
    base_dir = Path('L:\malle\CLM5_CH')
else:
    base_dir = Path('/home/lud11/malle/CLM5_CH')

bf_snow = base_dir/'snow_comp'
bf_meas = base_dir / 'dvd_oshd'

path_FSM = base_dir / 'FSM_new' / 'analysed_points'
all_files = glob.glob(os.path.join(path_FSM, "*.csv"))  # just do this once to get all ids

if platform.system() == 'Windows':
    all_locs_comp = list((f.split('\\')[-1]).split('_')[1] for f in all_files)
else:
    all_locs_comp = list((f.split('/')[-1]).split('_')[1] for f in all_files)

K = "MAE2"
K1 = "5DO"
all_locs = [i for i in all_locs_comp if (i != K and i != K1)]

all_elev = glob.glob(os.path.join(path_FSM, "*.csv"))
if platform.system() == 'Windows':
    all_names_elev = list((f.split('\\')[-1]).split('_')[1] for f in all_elev)
    all_elev_elev = list((f.split('\\')[-1]).split('_')[-1].split('.')[0] for f in all_elev)
else:
    all_names_elev = list((f.split('/')[-1]).split('_')[1] for f in all_elev)
    all_elev_elev = list((f.split('/')[-1]).split('_')[-1].split('.')[0] for f in all_elev)
all_elev_elev = list(map(int, all_elev_elev))
elev_comp = list(zip(all_names_elev, all_elev_elev))

max_all = []
elev_all = []
for locs_all in elev_comp:
        locs = locs_all[0]
        elev = locs_all[1]
        meas_in = pd.read_csv(glob.glob(os.path.join(bf_meas, "*" + locs + "*.csv"))[0]).set_index('time_HS_meas')
        meas_in = meas_in.loc[~meas_in.index.duplicated(keep='first')]  # this is necessary since some seasons overlapped..
        meas_in.set_index(pd.to_datetime(meas_in.index), inplace=True)
        meas_in.dropna(inplace=True)
        max_hs = np.max(meas_in['2017-10-01' :'2018-08-01'])
        max_all.append(max_hs)
        elev_all.append(elev)

df_max = pd.DataFrame(max_all, index=all_locs, columns=['snow_max'])
df_elev = pd.DataFrame(elev_all, index=all_locs, columns=['elev'])
df_all = pd.concat([df_max, df_elev], axis=1)
df_all.sort_values(by=['elev'], inplace=True)